# 02 — Fine-tuning PhoBERT cho Phân tích Cảm xúc

Notebook này thực hiện fine-tune **PhoBERT** (`vinai/phobert-base-v2`) cho bài toán
**Sentiment Classification** (Positive / Neutral / Negative) từ dữ liệu đánh giá sản phẩm tiếng Việt.

```
Load processed data → Dataset/DataLoader → Fine-tune PhoBERT (K-Fold)
    → Evaluate (Ensemble) → Save HuggingFace format → (Optional) Push to Hub
```

**Input:** `data/processed/*_labeled.csv` + `class_weights.json`  
**Output:** `models/phobert_sentiment/` (HuggingFace format, sẵn sàng deploy)


## 1. Khởi tạo môi trường

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install transformers datasets accelerate scikit-learn seaborn underthesea -q


In [ ]:
import os, json, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix,
)
warnings.filterwarnings('ignore')

# ── Cấu hình đường dẫn ────────────────────────────────────────
BASE_PATH  = '/content/drive/MyDrive/sentiment_analyst/'
DATA_PATH  = BASE_PATH + 'data/processed/'
MODEL_PATH = BASE_PATH + 'models/phobert_sentiment/'
CKPT_PATH  = BASE_PATH + 'outputs/checkpoints/'
LOG_PATH   = BASE_PATH + 'outputs/logs/'
FIG_PATH   = BASE_PATH + 'outputs/figures/'

for p in [DATA_PATH, MODEL_PATH, CKPT_PATH, LOG_PATH, FIG_PATH]:
    os.makedirs(p, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super(NpEncoder, self).default(obj)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False


## 2. Siêu tham số

In [ ]:
# ── Model ────────────────────────────────────────────────────
PHOBERT_NAME = 'vinai/phobert-base-v2'
NUM_LABELS   = 3
MAX_LEN      = 256

# ── Training ─────────────────────────────────────────────────
BATCH_SIZE       = 16      # Tránh OOM trên Colab T4
ACCUM_STEPS      = 8       # Effective BS = 128
NUM_WORKERS      = 0       # Colab không hỗ trợ multiprocess DataLoader ổn định
LEARNING_RATE    = 2e-5
NUM_EPOCHS       = 25
WARMUP_RATIO     = 0.1
MAX_GRAD_NORM    = 1.0
EARLY_STOP_PAT   = 7
DROPOUT_RATE     = 0.15

USE_AMP          = True
LLRD_FACTOR      = 0.95
USE_FOCAL_LOSS   = True
LABEL_SMOOTHING  = 0.1
K_FOLDS          = 5

# ── Label mapping ─────────────────────────────────────────────
LABEL_ENCODE = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
LABEL_DECODE = {v: k for k, v in LABEL_ENCODE.items()}
ID2LABEL     = LABEL_DECODE
LABEL2ID     = LABEL_ENCODE

print('Cấu hình:')
for k, v in {
    'Model': PHOBERT_NAME, 'MAX_LEN': MAX_LEN, 'BATCH_SIZE': BATCH_SIZE,
    'Effective BS': BATCH_SIZE * ACCUM_STEPS,
    'LR': LEARNING_RATE, 'EPOCHS': NUM_EPOCHS, 'EARLY_STOP': EARLY_STOP_PAT
}.items():
    print(f'  {k:15s}: {v}')


## 3. Tải dữ liệu

In [ ]:
train_df = pd.read_csv(DATA_PATH + 'train_labeled.csv')
val_df   = pd.read_csv(DATA_PATH + 'val_labeled.csv')
test_df  = pd.read_csv(DATA_PATH + 'test_labeled.csv')

for df in [train_df, val_df, test_df]:
    if 'label' not in df.columns:
        df['label'] = df['sentiment'].map(LABEL_ENCODE)
    assert df['label'].notna().all(), 'Label chứa NaN — kiểm tra cột sentiment!'
    assert df['label'].isin([0, 1, 2]).all(), 'Label nằm ngoài range [0, 1, 2]!'

full_train_df = pd.concat([train_df, val_df]).reset_index(drop=True)
n_dup = full_train_df.duplicated(subset=['Review']).sum()
if n_dup > 0:
    print(f'⚠️  Cảnh báo: {n_dup} review bị trùng lặp trong full_train!')

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'Các cột: {list(train_df.columns)}')
full_train_df[['Review', 'sentiment', 'label']].head(3)


In [ ]:
# Tính toán class_weights trên toàn bộ full_train để phản ánh đúng phân phối
full_labels = full_train_df['label'].values
cw_values = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=full_labels
)
class_weights = torch.tensor(cw_values, dtype=torch.float).to(device)
print('Class weights (recomputed on full_train):', cw_values.tolist())

# Vẫn load json để xác thực LABEL_ENCODE
with open(DATA_PATH + 'class_weights.json', encoding='utf-8') as f:
    cw_data = json.load(f)
assert cw_data['label_encode'] == LABEL_ENCODE, \
    f"Label encode không khớp!\nJSON: {cw_data['label_encode']}\nCode: {LABEL_ENCODE}"


## 4. Dataset & DataLoader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PHOBERT_NAME)

class PhoBERTDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        # Trở lại Lazy Tokenization trong __getitem__ để giảm thiểu RAM memory leak trên Colab
        self.reviews   = df['Review'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.reviews[idx],
            max_length      = self.max_len,
            padding         = 'max_length',
            truncation      = True,
            return_tensors  = 'pt',
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long),
        }

full_train_dataset = PhoBERTDataset(full_train_df, tokenizer, MAX_LEN)
test_dataset  = PhoBERTDataset(test_df,  tokenizer, MAX_LEN)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))

print(f'Full Train size: {len(full_train_dataset)} | Test: {len(test_dataset)}')

sample_reviews = full_train_df['Review'].sample(min(500, len(full_train_df)))
lengths = [len(tokenizer.encode(str(r))) for r in sample_reviews]
print(f'\nPhân tích độ dài Token (mẫu 500 reviews):')
print(f'  Mean={np.mean(lengths):.0f}, Median={np.median(lengths):.0f}')
print(f'  95th={np.percentile(lengths, 95):.0f}, Max={max(lengths)}')


## 5. Các hàm hỗ trợ

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=-1)
        pt = probs.gather(1, targets.view(-1, 1)).squeeze(1)
        ce_loss = nn.functional.cross_entropy(
            logits, targets, weight=self.weight, label_smoothing=self.label_smoothing, reduction='none'
        )
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()

def get_llrd_params(model, base_lr, factor):
    params = []
    params.append({'params': model.classifier.parameters(), 'lr': base_lr})
    layers = model.roberta.encoder.layer
    for i, layer in enumerate(reversed(layers)):
        lr = base_lr * (factor ** (i + 1))
        params.append({'params': layer.parameters(), 'lr': lr})
    params.append({'params': model.roberta.embeddings.parameters(), 'lr': base_lr * (factor ** len(layers))})
    return params

def train_epoch(model, loader, optimizer, scheduler, criterion, device, scaler):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for step, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        with autocast(device_type=device.type, enabled=USE_AMP):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            with autocast(device_type=device.type, enabled=USE_AMP):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss    = criterion(outputs.logits, labels)

            total_loss += loss.item()
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1, all_preds, all_labels


## 6. Huấn luyện với K-Fold Cross Validation


In [ ]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(full_train_df, full_train_df['label'])):
    print(f'\n' + '='*50)
    print(f'🚀 FOLD {fold + 1}/{K_FOLDS}')
    print('='*50)
    
    train_sub = Subset(full_train_dataset, train_idx)
    val_sub   = Subset(full_train_dataset, val_idx)
    
    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
    val_loader   = DataLoader(val_sub,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
    
    config = AutoConfig.from_pretrained(PHOBERT_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID)
    config.hidden_dropout_prob = DROPOUT_RATE
    config.attention_probs_dropout_prob = DROPOUT_RATE
    config.classifier_dropout = 0.3
    model = AutoModelForSequenceClassification.from_pretrained(PHOBERT_NAME, config=config).to(device)
    
    optimizer_grouped_parameters = get_llrd_params(model, LEARNING_RATE, LLRD_FACTOR)
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE, weight_decay=0.01)
    
    total_steps  = len(train_loader) * NUM_EPOCHS // ACCUM_STEPS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    if USE_FOCAL_LOSS:
        criterion = FocalLoss(weight=class_weights, gamma=2.0, label_smoothing=LABEL_SMOOTHING)
    else:
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    scaler = GradScaler('cuda', enabled=USE_AMP)
    
    best_val_f1 = 0.0
    early_stop_cnt = 0
    history = {'train_loss':[], 'train_f1':[], 'val_loss':[], 'val_f1':[], 'lr':[]}
    
    for epoch in range(1, NUM_EPOCHS + 1):
        current_lr = optimizer.param_groups[0]['lr']
        tr_loss, tr_acc, tr_f1 = train_epoch(model, train_loader, optimizer, scheduler, criterion, device, scaler)
        vl_loss, vl_acc, vl_f1, _, _ = evaluate(model, val_loader, criterion, device)
        
        history['train_loss'].append(tr_loss)
        history['train_f1'].append(tr_f1)
        history['val_loss'].append(vl_loss)
        history['val_f1'].append(vl_f1)
        history['lr'].append(current_lr)
        
        note = ''
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            early_stop_cnt = 0
            note = '✅ best'
            model.save_pretrained(CKPT_PATH + f'best_model_fold_{fold+1}')
            tokenizer.save_pretrained(CKPT_PATH + f'best_model_fold_{fold+1}')
        else:
            early_stop_cnt += 1
            note = f'EStop {early_stop_cnt}/{EARLY_STOP_PAT}'
        
        print(f'Epoch {epoch:>2} | LR={current_lr:.2e} | TrL={tr_loss:.4f} TrF1={tr_f1:.4f} | VlL={vl_loss:.4f} VlF1={vl_f1:.4f} | {note}')
        
        if early_stop_cnt >= EARLY_STOP_PAT:
            print(f'⏹ Early stopping Fold {fold+1} tại epoch {epoch}.')
            break
    
    print(f'✅ Fold {fold+1} Best Val F1: {best_val_f1:.4f}')
    fold_results.append(best_val_f1)
    
    with open(LOG_PATH + f'training_history_fold_{fold+1}.json', 'w', encoding='utf-8') as fout:
        json.dump(history, fout, indent=2, cls=NpEncoder)
        
    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n🏆 K-FOLD AVERAGE BEST VAL F1: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}')


## 7. Biểu đồ quá trình huấn luyện (Best Fold)

In [ ]:
if 'fold_results' in globals():
    best_fold = np.argmax(fold_results) + 1
    with open(LOG_PATH + f'training_history_fold_{best_fold}.json', 'r', encoding='utf-8') as f:
        history = json.load(f)
    epochs_range = range(1, len(history['train_loss']) + 1)
    best_epoch = np.argmax(history['val_f1']) + 1

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='#2196F3')
    axes[0].plot(epochs_range, history['val_loss'],   's-', label='Val Loss',   color='#F44336')
    axes[0].axvline(best_epoch, color='gray', linestyle='--', linewidth=1, label=f'Best epoch ({best_epoch})')
    axes[0].set_title(f'Loss (Fold {best_fold})', fontweight='bold', fontsize=13)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_range, history['train_f1'], 'o-', label='Train F1', color='#4CAF50')
    axes[1].plot(epochs_range, history['val_f1'],   's-', label='Val F1',   color='#FF9800')
    axes[1].axvline(best_epoch, color='gray', linestyle='--', linewidth=1, label=f'Best epoch ({best_epoch})')
    axes[1].set_title(f'F1 Macro (Fold {best_fold})', fontweight='bold', fontsize=13)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1 Score')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(f'PhoBERT Fine-tuning — Training Curves (Best Fold: {best_fold})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_PATH + 'training_curves.png', bbox_inches='tight', dpi=150)
    plt.show()


## 8. Đánh giá trên tập Test

In [ ]:
print('Tiến hành Ensemble 5 models trên Test Set...')

test_labels = []
with torch.no_grad():
    for batch in test_loader:
        test_labels.extend(batch['label'].numpy())
test_labels = np.array(test_labels)

all_logits = []
successful_folds = []
EVAL_DONE = False

for fold in range(1, K_FOLDS + 1):
    ckpt_path = CKPT_PATH + f'best_model_fold_{fold}'
    if not os.path.exists(ckpt_path):
        print(f'⚠️  Warning: Checkpoint for fold {fold} not found. Skipping.')
        continue
    
    try:
        fold_model = AutoModelForSequenceClassification.from_pretrained(ckpt_path).to(device)
        fold_model.eval()
        
        fold_logits = []
        with torch.no_grad():
            for batch in test_loader:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                with autocast(device_type=device.type, enabled=USE_AMP):
                    outputs = fold_model(input_ids=input_ids, attention_mask=attention_mask)
                fold_logits.append(outputs.logits.cpu())
                
        all_logits.append(torch.cat(fold_logits, dim=0))
        successful_folds.append(fold)
        print(f'✅ Fold {fold} loaded successfully')
        
        del fold_model
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f'❌ Error loading fold {fold}: {e}')
        continue

if not all_logits:
    print('❌ ERROR: Không có fold nào thành công!')
else:
    print(f'\n✅ Successfully loaded {len(successful_folds)} folds: {successful_folds}')
    
    all_probs = [torch.softmax(logits, dim=-1) for logits in all_logits]
    ensemble_probs = torch.stack(all_probs).mean(dim=0)
    test_preds = ensemble_probs.argmax(dim=-1).numpy()

    label_names = ['Negative', 'Neutral', 'Positive']
    print('\n=== KẾT QUẢ TRÊN TẬP TEST ===')
    print(f'Accuracy        : {accuracy_score(test_labels, test_preds):.4f}')
    print(f'F1 Macro        : {f1_score(test_labels, test_preds, average="macro", zero_division=0):.4f}')
    print(f'F1 Weighted     : {f1_score(test_labels, test_preds, average="weighted", zero_division=0):.4f}')
    print()
    print(classification_report(test_labels, test_preds, target_names=label_names, zero_division=0))


In [ ]:
if 'all_logits' in globals() and 'test_preds' in globals():
    cm      = confusion_matrix(test_labels, test_preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, data, fmt, title in [
        (axes[0], cm,      'd',    'Confusion Matrix (số lượng)'),
        (axes[1], cm_norm, '.2f', 'Confusion Matrix (% theo nhãn thực / Recall)'),
    ]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=label_names, yticklabels=label_names,
                    linewidths=0.5, ax=ax)
        ax.set_title(title, fontweight='bold', fontsize=12)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    plt.suptitle(f'PhoBERT — Confusion Matrix trên Test set (Ensemble K-Fold)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_PATH + 'confusion_matrix.png', bbox_inches='tight', dpi=150)
    plt.show()


In [ ]:
if 'all_logits' in globals() and 'test_preds' in globals():
    report_dict = classification_report(test_labels, test_preds, target_names=label_names, output_dict=True, zero_division=0)
    test_results = {
        'accuracy'   : float(accuracy_score(test_labels, test_preds)),
        'f1_macro'   : float(f1_score(test_labels, test_preds, average='macro',    zero_division=0)),
        'f1_weighted': float(f1_score(test_labels, test_preds, average='weighted', zero_division=0)),
        'precision_macro': float(precision_score(test_labels, test_preds, average='macro', zero_division=0)),
        'recall_macro'   : float(recall_score(test_labels, test_preds, average='macro', zero_division=0)),
        'ensemble_used'  : True,
        'per_class'  : report_dict,
    }
    with open(LOG_PATH + 'test_results.json', 'w', encoding='utf-8') as f:
        json.dump(test_results, f, indent=2, ensure_ascii=False, cls=NpEncoder)
    print('✅ Đã lưu kết quả vào test_results.json')
    EVAL_DONE = True


## 9. Lưu mô hình (Best Fold) để Deploy


In [ ]:
if 'EVAL_DONE' in globals() and EVAL_DONE:
    # Note: Lưu fold có val_f1 cao nhất. Trên thực tế có thể chọn fold có test_f1 cao nhất nếu không ensemble.
    best_fold = np.argmax(fold_results) + 1
    
    model_card = f"""---
language: vi
tags:
  - sentiment-analysis
  - vietnamese
  - phobert
  - text-classification
datasets:
  - custom-shopee-reviews
metrics:
  - f1
  - accuracy
pipeline_tag: text-classification
---

# PhoBERT Vietnamese Sentiment Analysis

Mô hình phân tích cảm xúc tiếng Việt fine-tuned từ [vinai/phobert-base-v2](https://huggingface.co/vinai/phobert-base-v2)
trên tập dữ liệu đánh giá sản phẩm thương mại điện tử.

## Nhãn
| Label | ID | Mô tả |
|-------|----|-----------|
| Negative | 0 | Đánh giá tiêu cực |
| Neutral  | 1 | Đánh giá trung tính / hỗn hợp |
| Positive | 2 | Đánh giá tích cực |

## Tiền xử lý bắt buộc
Văn bản **phải được tách từ** bằng `underthesea.word_tokenize(text, format='text')`
trước khi đưa vào model.

## Kết quả
| Metric | Score |
|--------|-------|
| Accuracy | {test_results['accuracy']:.4f} |
| F1 Macro | {test_results['f1_macro']:.4f} |
| F1 Weighted | {test_results['f1_weighted']:.4f} |
"""
    torch.cuda.empty_cache()
    
    best_model_to_save = AutoModelForSequenceClassification.from_pretrained(CKPT_PATH + f'best_model_fold_{best_fold}')
    best_model_to_save.save_pretrained(MODEL_PATH)
    tokenizer.save_pretrained(MODEL_PATH)
    del best_model_to_save
    gc.collect()

    with open(MODEL_PATH + 'README.md', 'w', encoding='utf-8') as f:
        f.write(model_card)
    print('✅ Model đã lưu tại:', MODEL_PATH)
else:
    print('⚠️  Warning: Chưa có kết quả test_results hoặc fold_results. Bỏ qua bước tạo Model Card.')


## 10. Demo Inference

In [ ]:
from underthesea import word_tokenize
import unicodedata, re

TEENCODE = {
    'ko':'không','k':'không','kh':'không','khong':'không',
    'đc':'được','dc':'được',
    'cx':'cũng','nma':'nhưng mà','sp':'sản phẩm',
    'mn':'mọi người','mng':'mọi người',
    'vl':'rất','vkl':'rất',
    'oke':'ok','okie':'ok','oki':'ok','okela':'ok','okila':'ok',
    'ship':'giao hàng','ib':'nhắn_tin','rep':'phản_hồi','feedback':'đánh_giá',
}
EMOJI = {
    '❤️':'tích_cực','🧡':'tích_cực','💛':'tích_cực','💚':'tích_cực',
    '💙':'tích_cực','💜':'tích_cực','🤍':'tích_cực','❣️':'tích_cực',
    '💗':'tích_cực','💓':'tích_cực','😍':'tích_cực','🥰':'tích_cực',
    '😊':'tích_cực','😄':'tích_cực','😁':'tích_cực','🤩':'tích_cực',
    '😻':'tích_cực','🌟':'tích_cực','⭐':'tích_cực','👍':'tích_cực',
    '💪':'tích_cực','✅':'tích_cực','👌':'tích_cực',
    '😡':'tiêu_cực','😠':'tiêu_cực','🤬':'tiêu_cực','😤':'tiêu_cực',
    '👎':'tiêu_cực','❌':'tiêu_cực','😭':'tiêu_cực','😢':'tiêu_cực','🥲':'tiêu_cực',
}

# ⚠️ QUAN TRỌNG: Hàm này PHẢI giống hệt hàm preprocess() trong notebook 01.
# Nếu sửa một bên, phải sửa cả bên kia.
def preprocess(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'@\w+', '', text)
    for emo, val in EMOJI.items(): text = text.replace(emo, f' {val} ')
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    words = [TEENCODE.get(w, w) for w in text.split()]
    text  = ' '.join(words)
    text  = re.sub(r'!+', '!', text)
    text  = re.sub(r'\?+', '?', text)
    text  = re.sub(r'[^\w\s!?_]', ' ', text)
    text  = re.sub(r'\s+', ' ', text).strip()
    
    if not text:
        return ''
        
    try:
        text = word_tokenize(text, format='text')
    except TypeError:
        try:
            result = word_tokenize(text)
            text = ' '.join(result) if isinstance(result, list) else result
        except Exception:
            pass
    return text

try:
    best_fold_id = np.argmax(fold_results) + 1
except NameError:
    print('⚠️  Warning: Không tìm thấy fold_results, fallback dùng fold 1. Có thể đây không phải fold tốt nhất.')
    best_fold_id = 1

ckpt_path = CKPT_PATH + f'best_model_fold_{best_fold_id}'
if not os.path.exists(ckpt_path):
    print(f'❌ ERROR: Checkpoint không tồn tại tại {ckpt_path}')
else:
    inference_model = AutoModelForSequenceClassification.from_pretrained(ckpt_path).to(device)
    print(f'✅ Đã load inference_model từ fold {best_fold_id}.')

def predict(text, model, tokenizer, device, threshold=0.4):
    model.eval()
    processed = preprocess(text)
    if not processed:
        return {'label': 'Unknown', 'confidence': 0.0, 'probs': {ID2LABEL[i]: 0.0 for i in range(NUM_LABELS)}, 'preprocessed': ''}
        
    try:
        enc = tokenizer(processed, max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')
        with torch.no_grad():
            outputs = model(input_ids=enc['input_ids'].to(device), attention_mask=enc['attention_mask'].to(device))
        probs      = torch.softmax(outputs.logits, dim=-1).cpu().squeeze()
        pred_id    = probs.argmax().item()
        confidence = probs[pred_id].item()
        
        if confidence < threshold:
            return {'label': 'Unknown', 'confidence': confidence, 'probs': {ID2LABEL[i]: round(p.item(), 4) for i, p in enumerate(probs)}, 'preprocessed': processed}
        return {'label': ID2LABEL[pred_id], 'confidence': confidence, 'probs': {ID2LABEL[i]: round(p.item(), 4) for i, p in enumerate(probs)}, 'preprocessed': processed}
    except Exception as e:
        return {'label': 'Error', 'confidence': 0.0, 'probs': {}, 'preprocessed': processed, 'error': str(e)}

test_cases = [
    'Giao hàng nhanh 👍, sản phẩm đẹp lắm mọi người nên mua!',
    'Chất lượng bình thường, giá hơi cao nhưng ship nhanh',
    'Hàng giả, chất lượng kém, giao hàng chậm. Không mua lại!',
    'shop nhiệt tình ko ngờ đẹp vl luôn 😍😍',
]
print('\n=== DEMO INFERENCE ===')
if 'inference_model' in globals():
    for text in test_cases:
        result = predict(text, inference_model, tokenizer, device)
        print(f'Input      : {text}')
        print(f'Preprocessed: {result["preprocessed"]}')
        print(f'Kết quả    : {result["label"]} (confidence: {result["confidence"]:.4f})')
        print(f'Xác suất   : {result["probs"]}')
        if 'error' in result:
            print(f'⚠️  Error: {result["error"]}')
        print()
